<a href="https://colab.research.google.com/github/24sahil25-source/BA305-BoardGameGeek-Fall2026/blob/main/notebooks/13_final_comparison_ansel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 10: compare the tree, k-NN and the neural network on the final rows, choose one to deploy, show what drives it, and hand it to task 3

Tasks 1 and 2 are in. Notebook 05 settled the rows, the split, the inputs and the imputer, and tuned the tree and k-NN. Notebook 06 tuned a neural network on the same rows. Notebook 08 (task 4) decided that year of publication stays an input. This notebook puts the three methods side by side on one footing and answers four questions:

1. How do the tree, k-NN and the network compare on the same rows, split, folds and score?
2. Which one would we deploy, and why?
3. What drives that model?
4. What does task 3's cost-based decision look like with that model, and what does notebook 07 need to paste in to re-run on it?

Every choice below was made by Ansel Weng before any code was written:

| # | Decision | Choice |
| --- | --- | --- |
| 1 | Rows, split, inputs, imputer | Notebook 05's section 13 cell, pasted unchanged: 21,379 games, 14,964 training and 6,415 test, 45 inputs, `SimpleImputer(strategy="median")` as the first step of every pipeline |
| 2 | Settings | Not re-tuned. Tree `max_depth=12, min_samples_leaf=50` (notebook 05); k-NN `n_neighbors=30, weights="distance", p=1` with `StandardScaler` (notebook 05); network `(2, 2, 2)`, logistic, `lbfgs`, `tol=1e-8`, `max_iter=2000`, `MinMaxScaler`, rating scaled to 0–1 by `(rating - 1) / 9` (notebook 06) |
| 3 | Cross-validation | All three re-scored on the same `KFold(n_splits=5, shuffle=True, random_state=0)` used in 05 and 06, so their numbers sit on one footing |
| 4 | Network seeds | Seeds 1 to 5, as in notebook 06. In the deploy rule the network is represented by the **mean over seeds of each seed's CV RMSE** (notebook 06's number). The CV RMSE of the 5-seed average is shown for information only |
| 5 | Deploy rule | Lowest CV RMSE on the training folds. Any model within **0.005 RMSE** of the best counts as tied, and the **simplest tied model wins**: tree < k-NN < network. This is notebook 05's tie rule. The test rows are reported, never used to choose |
| 6 | Deployed network, if it wins | The average of the 5 seeds' predictions, so no single lucky (or stuck) seed decides it |
| 7 | References | `DummyRegressor` (ground rule 3) and linear regression (notebook 06's reference) |
| 8 | Test sets | All 6,415 test games (headline), the 5,395 complete games (notebook 04's test rows) and the 1,020 games with a missing input, as in notebook 05 |
| 9 | What drives the model | Permutation importance on the test rows for all three models, so they are compared on one footing; the tree's own `feature_importances_` if the tree is deployed (Session 19's method); and predicted against actual rating across the top inputs |
| 10 | Task 3 | Notebook 07 belongs to Andrew (ground rule 6), so it is not edited here. Section 9 re-runs task 3's cutoff and cost steps on the deployed model as a check, and section 10 gives the cell that notebook 07 pastes in |

Ground rules: same loading cell as notebooks 02 to 06; RMSE with MAE and R² next to `DummyRegressor`; no post-release column (`rank`, `bayes_average`, `users_rated`, `owned`, `trading`, `wanting`, `wishing`) as an input; every imputer and scaler fitted on training rows only, inside a `Pipeline`; the test rows scored once.

In [ ]:
# --- Ansel Weng: task 10, compare the tree, k-NN and the neural network on the final rows, choose one to deploy, show what drives it, and hand it to task 3 ---
# [AI-assisted] Claude Code, 2026-11-11. Every code and markdown cell in this notebook was generated with Claude Code from Ansel Weng's decisions, notebooks 05 to 08 and the course labs; reviewed and edited by Ansel Weng.
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, precision_score, recall_score
from sklearn.exceptions import ConvergenceWarning

warnings.simplefilter("ignore", ConvergenceWarning)  # the network's solver warns when it hits max_iter; we count those fits ourselves, as notebook 06 does

## 1. Load the data, and rebuild task 1's rows, split and inputs

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Same loading cell as notebooks 02 to 06, and the figure folder helper from notebook 05; reviewed and edited by Ansel Weng.
BASE = "https://raw.githubusercontent.com/24sahil25-source/BA305-BoardGameGeek-Fall2026/main/data/"
games = pd.read_csv(BASE + "bgg_clean.csv")
print(games.shape)

FIG_DIR = "../figures" if os.path.isdir("../figures") else "."

def save(name):
    plt.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches="tight")

**Task 1's cell, pasted unchanged** from notebook 05, section 13. Notebook 06 pasted the same cell, so the three models see exactly the same 14,964 training and 6,415 test games and the same 45 inputs. The checks after the paste confirm the counts and that none of the post-release columns is an input.

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Notebook 05's section 13 cell (rows, split and inputs), pasted unchanged, followed by checks; reviewed and edited by Ansel Weng.
# ---- paste from here: rows, split, inputs and imputer as settled by task 1 (notebooks/05_imputation_tuning_ansel.ipynb) ----
numeric_cols = ["yearpublished", "minplayers", "maxplayers", "minplaytime", "maxplaytime", "minage", "n_mechanics", "n_categories"]
gaps = games[numeric_cols].isna().sum(axis=1)
data = games[games["yearpublished"].notna() & (gaps < 4)].copy()           # drop no-year games and games missing 4+ of the 8 inputs
data["complete"] = data[numeric_cols].notna().all(axis=1)
tr_c, te_c = train_test_split(data[data["complete"]].reset_index(drop=True), test_size=0.3, random_state=1)    # = notebook 04's split
tr_i, te_i = train_test_split(data[~data["complete"]].reset_index(drop=True), test_size=0.3, random_state=1)
train_df, test_df = pd.concat([tr_c, tr_i], ignore_index=True), pd.concat([te_c, te_i], ignore_index=True)
y_train, y_test = train_df["average"], test_df["average"]

def build_inputs(frame, mechanics, categories):
    X = frame[numeric_cols].copy()
    for col in ["minplayers", "maxplayers", "minplaytime", "maxplaytime"]:
        X[col] = np.log10(X[col])
    mech = frame["boardgamemechanic"].str.get_dummies(sep="|").reindex(columns=mechanics, fill_value=0).add_prefix("mech_")
    cat = frame["boardgamecategory"].str.get_dummies(sep="|").reindex(columns=categories, fill_value=0).add_prefix("cat_")
    flags = pd.DataFrame({"mech_list_missing": frame["boardgamemechanic"].isna().astype(int),
                          "cat_list_missing": frame["boardgamecategory"].isna().astype(int)}, index=frame.index)
    return pd.concat([X, mech, cat, flags], axis=1)

top_mech = train_df["boardgamemechanic"].str.get_dummies(sep="|").mean().sort_values(ascending=False).head(20).index
top_cat = train_df["boardgamecategory"].str.get_dummies(sep="|").mean().sort_values(ascending=False).head(15).index
X_train, X_test = build_inputs(train_df, top_mech, top_cat), build_inputs(test_df, top_mech, top_cat)
# ---- paste to here ----

test_complete = test_df["complete"].values
post_release = ["rank", "bayes_average", "users_rated", "owned", "trading", "wanting", "wishing"]
assert (len(X_train), len(X_test), X_train.shape[1]) == (14964, 6415, 45), "rows or inputs differ from notebooks 05 and 06"
assert not set(post_release) & set(X_train.columns)
print("training games:", len(X_train), "| test games:", len(X_test), f"({test_complete.sum():,} complete, {(~test_complete).sum():,} with a gap) | inputs:", X_train.shape[1])
print("post-release columns among the inputs:", [c for c in post_release if c in X_train.columns])

## 2. The three models, at the settings notebooks 05 and 06 chose

Decision 2: nothing is re-tuned. Each model is a `Pipeline` that starts with task 1's median imputer, so in every cross-validation fold the imputer and scaler are fitted on the four training folds only.

- **Tree and k-NN** are notebook 05's pipelines, built the same way (`imputer_steps` + `model_steps` in notebook 05).
- **The network** is notebook 06's pipeline: median imputer, `MinMaxScaler`, then `MLPRegressor`. Notebook 06 scaled the rating to 0–1 with `to_scaled` and converted predictions back with `to_ratings`. Here the same two functions are handed to scikit-learn's `TransformedTargetRegressor`, which applies them around `fit` and `predict`. The arithmetic is identical; the only difference is that the network can then be scored, cross-validated and permuted with the same code as the tree and k-NN.
- **`SeedAverage`** fits the network once per seed (1 to 5) and averages the five predictions. It is the deployed form of the network if the network wins (decision 6).

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. The three tuned pipelines, the two references and the 5-seed network average; reviewed and edited by Ansel Weng.
SEEDS = [1, 2, 3, 4, 5]
MAX_ITER = 2000

def to_scaled(ratings):          # notebook 06's rating scaling: the 1-10 scale's own limits, nothing learned from the data
    return (np.asarray(ratings) - 1) / 9

def to_ratings(scaled):          # notebook 06's inverse, the lab's to_euros adapted to ratings
    return np.asarray(scaled) * 9 + 1

def make_tree():                 # notebook 05, section 7: depth 12, at least 50 games per leaf
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("model", DecisionTreeRegressor(max_depth=12, min_samples_leaf=50, random_state=0))])

def make_knn():                  # notebook 05, section 8: k = 30, distance weights, Manhattan distance
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()),
                     ("model", KNeighborsRegressor(n_neighbors=30, weights="distance", p=1))])

def make_network(seed):          # notebook 06, Part 4: (2, 2, 2) logistic, lbfgs, the lab's tol and max_iter fix
    net = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", MinMaxScaler()),
                    ("model", MLPRegressor(hidden_layer_sizes=(2, 2, 2), activation="logistic", solver="lbfgs",
                                           tol=1e-8, max_iter=MAX_ITER, random_state=seed))])
    return TransformedTargetRegressor(regressor=net, func=to_scaled, inverse_func=to_ratings, check_inverse=False)

class SeedAverage(RegressorMixin, BaseEstimator):
    """The network fitted once per seed; predicts the average of the seeds' predictions."""
    def __init__(self, seeds=(1, 2, 3, 4, 5)):
        self.seeds = seeds

    def fit(self, X, y):
        self.networks_ = [make_network(seed).fit(X, y) for seed in self.seeds]
        return self

    def predict(self, X):
        return np.mean([net.predict(X) for net in self.networks_], axis=0)

def make_reference(name):
    model = DummyRegressor(strategy="mean") if name == "baseline" else LinearRegression()
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("model", model)])

MODELS = ["tree", "k-NN", "network"]                       # simplest first: the order decision 5 breaks ties by
make = {"tree": make_tree, "k-NN": make_knn}

## 3. Cross-validated RMSE on the same folds

Every model is scored on the same 5 folds of the 14,964 training games. The network is cross-validated once per seed. As in notebooks 05 and 06, the CV RMSE is the mean of the five fold RMSEs.

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. 5-fold cross-validated RMSE for the references, the tree, k-NN and the network with each seed; reviewed and edited by Ansel Weng.
folds = KFold(n_splits=5, shuffle=True, random_state=0)

def cv_rmse(model):
    return -cross_val_score(model, X_train, y_train, cv=folds, scoring="neg_root_mean_squared_error", n_jobs=-1)

start = time.time()
cv_folds = {name: cv_rmse(make_reference(name)) for name in ["baseline", "linear regression"]}
cv_folds["tree"] = cv_rmse(make_tree())
cv_folds["k-NN"] = cv_rmse(make_knn())
network_cv = pd.DataFrame({seed: cv_rmse(make_network(seed)) for seed in SEEDS})      # rows: folds, columns: seeds
network_cv.index = [f"fold {i + 1}" for i in range(5)]
print(f"cross-validation took {time.time() - start:.0f} seconds")
network_cv.round(4).assign(**{"mean over seeds": network_cv.mean(axis=1)}).T.assign(**{"CV RMSE": lambda d: d.mean(axis=1)}).round(4)

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. For information only: CV RMSE of the 5-seed average, the form the network would be deployed in (decision 4); reviewed and edited by Ansel Weng.
start = time.time()
seed_average_cv = cv_rmse(SeedAverage(seeds=SEEDS))
print(f"5-seed average of the network: CV RMSE {seed_average_cv.mean():.4f} (sd over folds {seed_average_cv.std():.4f}), {time.time() - start:.0f} seconds")

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. One cross-validation table for every model; reviewed and edited by Ansel Weng.
per_seed = network_cv.mean(axis=0)              # each seed's CV RMSE
cv_table = pd.DataFrame({name: {"CV RMSE": s.mean(), "sd over folds": s.std(), "sd over seeds": np.nan} for name, s in cv_folds.items()}).T
cv_table.loc["network"] = {"CV RMSE": per_seed.mean(), "sd over folds": network_cv.mean(axis=1).std(), "sd over seeds": per_seed.std()}
cv_table.loc["network, 5-seed average (information only)"] = {"CV RMSE": seed_average_cv.mean(), "sd over folds": seed_average_cv.std(), "sd over seeds": np.nan}
cv_table["notebook 05 or 06 reported"] = pd.Series({"tree": 0.6977, "k-NN": 0.7277, "network": 0.6929, "baseline": 0.9222, "linear regression": 0.7457})
cv_table.round(4)

## 4. Which model to deploy

Decision 5 applied: the best CV RMSE among the three methods, a 0.005 margin, and the simplest tied model wins.

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Decision 5 applied: tied = within 0.005 of the best CV RMSE; the simplest tied model is deployed; reviewed and edited by Ansel Weng.
MARGIN = 0.005
method_cv = cv_table.loc[MODELS, "CV RMSE"]
gap = method_cv - method_cv.min()
tied = gap <= MARGIN
DEPLOY = next(name for name in MODELS if tied[name])
display(pd.DataFrame({"CV RMSE": method_cv, "gap to best": gap, "tied with best": tied}).round(4))
print("best CV RMSE:", method_cv.idxmin(), f"({method_cv.min():.4f})")
print("tied with it:", list(tied[tied].index))
print("deployed (simplest tied model):", DEPLOY)

TAKEAWAY_DEPLOY

## 5. Score the test rows once

Each model is fitted on all 14,964 training games and scored once on the test games. The network is fitted with each seed (reported as mean and spread, as notebook 06 does) and as the 5-seed average (the form it would be deployed in).

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Fit every model on the training games and score each test set once; reviewed and edited by Ansel Weng.
def metrics(y, pred):
    return {"games": len(y), "RMSE": np.sqrt(mean_squared_error(y, pred)), "MAE": mean_absolute_error(y, pred), "R2": r2_score(y, pred)}

start = time.time()
fitted = {"baseline": make_reference("baseline").fit(X_train, y_train),
          "linear regression": make_reference("linear regression").fit(X_train, y_train),
          "tree": make_tree().fit(X_train, y_train),
          "k-NN": make_knn().fit(X_train, y_train),
          "network, 5-seed average": SeedAverage(seeds=SEEDS).fit(X_train, y_train)}
seed_nets = dict(zip(SEEDS, fitted["network, 5-seed average"].networks_))   # the five single-seed networks inside the average
print(f"fitting took {time.time() - start:.0f} seconds")

steps = {seed: net.regressor_.named_steps["model"].n_iter_ for seed, net in seed_nets.items()}
print("network training steps per seed:", steps, "| hit max_iter:", [s for s, n in steps.items() if n >= MAX_ITER])

test_sets = {"all test games": np.ones(len(y_test), dtype=bool), "complete test games": test_complete,
             "test games with a missing input": ~test_complete}
test_pred = {name: model.predict(X_test) for name, model in fitted.items()}
seed_pred = {seed: net.predict(X_test) for seed, net in seed_nets.items()}

rows = []
for set_name, mask in test_sets.items():
    for name, pred in test_pred.items():
        rows.append({"test set": set_name, "model": name, **metrics(y_test[mask], pred[mask])})
    per_seed_scores = pd.DataFrame([metrics(y_test[mask], p[mask]) for p in seed_pred.values()])
    rows.append({"test set": set_name, "model": "network, mean of 5 seeds", **per_seed_scores.mean().to_dict()})
    rows.append({"test set": set_name, "model": "network, sd over 5 seeds", **per_seed_scores.std().drop("games").to_dict()})
order = ["baseline", "linear regression", "tree", "k-NN", "network, mean of 5 seeds", "network, sd over 5 seeds", "network, 5-seed average"]
final = pd.DataFrame(rows).set_index(["test set", "model"]).reindex(pd.MultiIndex.from_product([list(test_sets), order]))
final["games"] = final["games"].astype("Int64")
final.round(3)

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Chart: CV RMSE with fold spread next to test RMSE, every model; reviewed and edited by Ansel Weng.
chart = pd.DataFrame({
    "CV RMSE (training folds)": [cv_table.loc[n, "CV RMSE"] for n in ["baseline", "linear regression", "tree", "k-NN", "network"]],
    "test RMSE (all 6,415 test games)": [final.loc[("all test games", n), "RMSE"] for n in ["baseline", "linear regression", "tree", "k-NN", "network, mean of 5 seeds"]],
}, index=["baseline", "linear\nregression", "tree", "k-NN", "network\n(mean of 5 seeds)"])
errors = pd.DataFrame({
    "CV RMSE (training folds)": [cv_table.loc[n, "sd over folds"] for n in ["baseline", "linear regression", "tree", "k-NN"]] + [cv_table.loc["network", "sd over seeds"]],
    "test RMSE (all 6,415 test games)": [0, 0, 0, 0, final.loc[("all test games", "network, sd over 5 seeds"), "RMSE"]],
}, index=chart.index)
ax = chart.plot(kind="bar", yerr=errors, capsize=3, rot=0, figsize=(10, 4.8), color=["0.6", "0.2"])
ax.set_ylim(0.6, 0.97)
ax.set_title(f"The tree, k-NN and the network on the same rows (deployed: {DEPLOY})")
ax.set_xlabel("")
ax.set_ylabel("RMSE (rating points), lower is better")
ax.legend(loc="upper right")
for i, v in enumerate(chart.iloc[:, 1]):
    ax.text(i + 0.13, v + 0.004, f"{v:.3f}", ha="center", fontsize=8)
save("13_model_comparison.png")
plt.show()
print("Error bars: sd over the 5 folds for CV; for the network, sd over the 5 seeds.")

TAKEAWAY_TEST

### Diagnosing the fit

Training RMSE (each model scored on the games it was fitted on) against CV RMSE and test RMSE. A training error far below the other two is the overfitting signal. k-NN with distance weights scores each training game against itself (notebook 05, section 8b), so its training RMSE is near zero by construction and says nothing about fit.

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Training, CV and test RMSE side by side for every model; reviewed and edited by Ansel Weng.
fit_rows = []
for name, cv_name, test_name in [("linear regression", "linear regression", "linear regression"), ("tree", "tree", "tree"),
                                 ("k-NN", "k-NN", "k-NN"), ("network, 5-seed average", "network", "network, 5-seed average")]:
    train_rmse = np.sqrt(mean_squared_error(y_train, fitted[name].predict(X_train)))
    fit_rows.append({"model": name, "training RMSE": train_rmse, "CV RMSE": cv_table.loc[cv_name, "CV RMSE"],
                     "test RMSE": final.loc[("all test games", test_name), "RMSE"]})
fit_table = pd.DataFrame(fit_rows).set_index("model")
fit_table["test - training"] = fit_table["test RMSE"] - fit_table["training RMSE"]
fit_table.round(3)

TAKEAWAY_FIT

## 6. Where the three models agree and disagree

Predicted against actual rating on the test games, one panel per model, and the correlation between the models' predictions. If the models make the same mistakes, the choice between them matters less than the inputs they share.

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Predicted vs actual for each model, and how closely their predictions agree; reviewed and edited by Ansel Weng.
panels = [("tree", "tree"), ("k-NN", "k-NN"), ("network, 5-seed average", "network (5-seed average)")]
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
for ax, (name, label) in zip(axes, panels):
    ax.scatter(y_test, test_pred[name], s=4, alpha=0.25, color="0.3")
    ax.plot([1, 10], [1, 10], color="red", linestyle="--", label="perfect prediction")
    ax.set_title(f"{label}: test RMSE {final.loc[('all test games', name), 'RMSE']:.3f}")
    ax.set_xlabel("Actual average rating")
    ax.legend(loc="upper left")
axes[0].set_ylabel("Predicted average rating")
fig.suptitle(f"Predicted vs actual on the {len(y_test):,} test games: all three pull predictions towards the middle")
plt.tight_layout()
save("13_predicted_vs_actual.png")
plt.show()

agree = pd.DataFrame({label: test_pred[name] for name, label in panels})
print("correlation between the models' test predictions:")
agree.corr().round(3)

TAKEAWAY_AGREE

## 7. What drives the models

**Permutation importance** (decision 9): shuffle one input's values across the test games, so it carries no information, and measure how much the test RMSE rises. A large rise means the model relies on that input. It treats every model the same way, which the tree's built-in importance cannot do, and it is measured on games the models never saw. Each input is shuffled 10 times; the bars show the mean rise and the error bars its spread.

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Permutation importance on the test games for the tree, k-NN and the network; reviewed and edited by Ansel Weng.
start = time.time()
perm = {}
for name, label in panels:
    result = permutation_importance(fitted[name], X_test, y_test, scoring="neg_root_mean_squared_error",
                                    n_repeats=10, random_state=0, n_jobs=-1)
    perm[label] = pd.DataFrame({"rise in RMSE": result.importances_mean, "sd": result.importances_std}, index=X_test.columns)
print(f"permutation importance took {time.time() - start:.0f} seconds")

deployed_label = dict(panels)[{"tree": "tree", "k-NN": "k-NN", "network": "network, 5-seed average"}[DEPLOY]]
rank_order = perm[deployed_label]["rise in RMSE"].sort_values(ascending=False).index
perm_table = pd.DataFrame({label: p["rise in RMSE"] for label, p in perm.items()}).loc[rank_order]
perm_table.head(12).round(3)

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Chart: the 12 inputs the deployed model relies on most, with the other two models alongside; reviewed and edited by Ansel Weng.
top = list(rank_order[:12])
fig, ax = plt.subplots(figsize=(9, 6))
height = 0.27
colours = {"tree": "0.15", "k-NN": "0.55", "network (5-seed average)": "tab:orange"}
for j, label in enumerate(perm):
    ax.barh(np.arange(len(top)) + (j - 1) * height, perm[label].loc[top, "rise in RMSE"], height=height,
            xerr=perm[label].loc[top, "sd"], color=colours[label], label=label + (" (deployed)" if label == deployed_label else ""))
ax.set_yticks(np.arange(len(top)))
ax.set_yticklabels(top)
ax.invert_yaxis()
ax.set_title("What the models rely on: rise in test RMSE when one input is shuffled")
ax.set_xlabel("Rise in test RMSE (rating points), mean of 10 shuffles")
ax.legend(loc="lower right")
save("13_permutation_importance.png")
plt.show()

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. The tree's own feature importance, Session 19's method, next to its permutation importance; reviewed and edited by Ansel Weng.
tree_importance = pd.Series(fitted["tree"].named_steps["model"].feature_importances_, index=X_train.columns)
both = pd.DataFrame({"built-in importance (share of squared-error reduction, training games)": tree_importance,
                     "permutation importance (rise in test RMSE)": perm["tree"]["rise in RMSE"]})
both = both.sort_values(both.columns[0], ascending=False)
if DEPLOY == "tree":
    ax = tree_importance.sort_values().tail(12).plot.barh(color="0.4", figsize=(8, 5))
    ax.set_title("Deployed tree: 12 most used inputs (built-in importance)")
    ax.set_xlabel("Share of squared-error reduction")
    save("13_tree_feature_importance.png")
    plt.show()
both.head(12).round(3)

TAKEAWAY_DRIVERS

### What the top inputs do to the prediction

Importance says *how much* the deployed model relies on an input, not *in which direction*. As in Session 19's notebook, the test games are grouped by each of the three most important inputs, and the average predicted rating is set against the average actual rating in each group. Play times and player counts are shown in their original units (the model sees their log10).

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Average predicted and actual rating across the deployed model's three most important inputs; reviewed and edited by Ansel Weng.
deployed_pred = test_pred[{"tree": "tree", "k-NN": "k-NN", "network": "network, 5-seed average"}[DEPLOY]]
check = test_df.copy()
check["predicted"] = deployed_pred

def groups_for(col):
    # Groups of test games for one input; games with that input missing are left out of its panel.
    if col == "yearpublished":
        return pd.cut(X_test[col], bins=[-np.inf, 1969, 1979, 1989, 1999, 2009, 2014, 2019, np.inf],
                      labels=["before 1970", "1970s", "1980s", "1990s", "2000-09", "2010-14", "2015-19", "2020+"])
    if col in ["minplayers", "maxplayers", "minplaytime", "maxplaytime"]:
        return pd.qcut(10 ** X_test[col], q=6, duplicates="drop")          # back to minutes or players
    if X_test[col].dropna().isin([0, 1]).all():
        return X_test[col].map({0: "no", 1: "yes"})                       # 0/1 mechanic, category or flag column
    return pd.qcut(X_test[col], q=6, duplicates="drop")

top3 = list(rank_order[:3])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True)
for ax, col in zip(axes, top3):
    g = check.groupby(groups_for(col).values, observed=True)[["average", "predicted"]].mean()
    n = check.groupby(groups_for(col).values, observed=True).size()
    x = np.arange(len(g))
    ax.plot(x, g["average"], "o-", color="black", label="actual")
    ax.plot(x, g["predicted"], "o--", color="tab:orange", label=f"predicted ({DEPLOY})")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{i}\n(n={c:,})" for i, c in zip(g.index.astype(str), n)], fontsize=7, rotation=30)
    ax.set_title(col)
    ax.set_xlabel(col + (", original units" if col in ["minplayers", "maxplayers", "minplaytime", "maxplaytime"] else ""))
axes[0].set_ylabel("Average rating of the test games in the group")
axes[0].legend()
fig.suptitle(f"Deployed {DEPLOY}: average predicted and actual rating across its three most important inputs (test games)")
plt.tight_layout()
save("13_top_inputs_effect.png")
plt.show()

TAKEAWAY_EFFECT

## 8. Leakage check on the top inputs

Directives §5.3: a top input that makes no sense should be checked for leakage. Every input the deployed model leans on is known before release: year of publication, play time, player counts, minimum age, the publisher's listed mechanics and categories. The one input that could be partly post-release, `mech_list_missing` (an obscure game is less likely to get its mechanics filled in on BoardGameGeek), was flagged by notebook 05. Its importance is printed below.

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Where the two list flags rank, for each model; reviewed and edited by Ansel Weng.
flag_rank = pd.DataFrame({label: p["rise in RMSE"].rank(ascending=False).astype(int) for label, p in perm.items()}).loc[["mech_list_missing", "cat_list_missing"]]
flag_size = pd.DataFrame({label: p["rise in RMSE"] for label, p in perm.items()}).loc[["mech_list_missing", "cat_list_missing"]]
pd.concat({"rank among 45 inputs": flag_rank, "rise in test RMSE": flag_size.round(4)}, axis=1)

TAKEAWAY_LEAK

## 9. Task 3 re-run on the deployed model (check)

Notebook 07 (Andrew) turns the predicted rating into a production decision, with notebook 04's tree on notebook 04's rows. This section re-runs its sections 3 to 6 unchanged in method, with the deployed model and task 1's rows, so the numbers are ready and section 10's paste cell is tested. Everything is notebook 07's:

- **Hit:** actual rating of 7.0 or more. **Flop** (made, not a hit) costs 3; **missed hit** (not made, was a hit) costs 1.
- The cutoff is chosen on **validation rows** (30% of the training games, `random_state=1`) from 6.0 to 8.5 in steps of 0.1, then scored once on the test games next to making every game or none.
- Other cost ratios: 1:3, 1:1, 2:1, 3:1 and 5:1.

The test set now has 6,415 games instead of 5,395, so the total costs are larger than notebook 07's and are compared within this table, not with notebook 07's.

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Notebook 07's hit line, cost function and validation split, with the deployed model; reviewed and edited by Ansel Weng.
make["network"] = lambda: SeedAverage(seeds=SEEDS)
make_model = make[DEPLOY]                   # notebook 07's MODEL hook

HIT = 7.0

def decision_cost(produce, hit, flop_cost=3, miss_cost=1):     # notebook 07, section 3, unchanged
    produce = np.asarray(produce)
    hit = np.asarray(hit)
    flops = ((produce == 1) & (hit == 0)).sum()
    missed = ((produce == 0) & (hit == 1)).sum()
    return flop_cost * flops + miss_cost * missed

hit_test = np.where(y_test >= HIT, 1, 0)
X_fit, X_valid, y_fit, y_valid = train_test_split(X_train, y_train, test_size=0.3, random_state=1)
hit_valid = np.where(y_valid >= HIT, 1, 0)
pred_valid = make_model().fit(X_fit, y_fit).predict(X_valid)
pred_test = deployed_pred                    # the deployed model, fitted on all training games in section 5

cutoff_grid = np.round(np.arange(6.0, 8.51, 0.1), 1)
costs = [decision_cost(np.where(pred_valid >= c, 1, 0), hit_valid) for c in cutoff_grid]
best_cut = cutoff_grid[int(np.argmin(costs))]
print("share of hits: validation rows", round(hit_valid.mean(), 3), "| test games", round(hit_test.mean(), 3))
print(f"deployed {DEPLOY}: lowest validation cost {min(costs)} at cutoff {best_cut} (notebook 07's tree on notebook 04's rows: 7.3)")

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Notebook 07's policy comparison on the test games, with the deployed model; reviewed and edited by Ansel Weng.
policies = {f"Model, cutoff {best_cut} (chosen)": np.where(pred_test >= best_cut, 1, 0),
            f"Model, naive cutoff {HIT}": np.where(pred_test >= HIT, 1, 0),
            "Make every game": np.ones(len(hit_test), dtype=int),
            "Make no game": np.zeros(len(hit_test), dtype=int)}
rows = []
for name, produce in policies.items():
    rows.append({"policy": name, "total cost": decision_cost(produce, hit_test), "games made": int(produce.sum()),
                 "flops made": int(((produce == 1) & (hit_test == 0)).sum()), "hits missed": int(((produce == 0) & (hit_test == 1)).sum()),
                 "precision": precision_score(hit_test, produce, zero_division=0), "recall": recall_score(hit_test, produce)})
policy_table = pd.DataFrame(rows).set_index("policy").sort_values("total cost")
policy_table.round(3)

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Notebook 07's cost-ratio sensitivity, with the deployed model; reviewed and edited by Ansel Weng.
rows = []
for flop_cost, miss_cost in [(1, 3), (1, 1), (2, 1), (3, 1), (5, 1)]:
    costs_r = [decision_cost(np.where(pred_valid >= c, 1, 0), hit_valid, flop_cost, miss_cost) for c in cutoff_grid]
    cut_r = cutoff_grid[int(np.argmin(costs_r))]
    model_cost = decision_cost(np.where(pred_test >= cut_r, 1, 0), hit_test, flop_cost, miss_cost)
    all_cost = decision_cost(policies["Make every game"], hit_test, flop_cost, miss_cost)
    none_cost = decision_cost(policies["Make no game"], hit_test, flop_cost, miss_cost)
    rows.append({"flop : missed hit": f"{flop_cost} : {miss_cost}", "chosen cutoff": cut_r, "model": model_cost,
                 "make every game": all_cost, "make no game": none_cost,
                 "model saves vs best no-model policy": 1 - model_cost / min(all_cost, none_cost)})
pd.DataFrame(rows).set_index("flop : missed hit").round(3)

TAKEAWAY_TASK3

## 10. The cell for notebook 07

Notebook 07 marks two cells to replace: **ROWS** (its section 1 and 2 cells: rows, split and `build_inputs`) and **MODEL** (`make_model()`). The cell below replaces both. Paste it in place of notebook 07's cells 5, 6 and 8 (the loading cell stays). It is run here and checked against this notebook's rows, inputs and predictions.

Three notes for notebook 07 once it is pasted:

1. **Inputs now have gaps** (the median imputer fills them inside the model's pipeline). Notebook 07's section 7 classifier is fitted on `X_train` directly. scikit-learn's trees accept missing values, so it will run, but ground rule 5 asks for the imputer inside the pipeline: `Pipeline([("impute", SimpleImputer(strategy="median")), ("model", DecisionTreeClassifier(...))])`, with `model__` in front of the grid's keys.
2. **Some test games have no category list.** In section 8, `check["boardgamecategory"].str.contains("Wargame")` returns a missing value for them, which `np.where` counts as a wargame; add `na=False`.
3. The test set grows from 5,395 to 6,415 games, so total costs are not comparable with the earlier run.

In [ ]:
# [AI-assisted] Claude Code, 2026-11-11. Copy-paste cell for notebook 07, checked against this notebook; reviewed and edited by Ansel Weng.
saved = (X_train.copy(), X_test.copy(), test_df["id"].copy())

# ---- paste into notebook 07 from here: ROWS and MODEL, as settled by tasks 1 and 10 (notebooks 05 and 13) ----
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ROWS: task 1's rows, split and inputs (notebook 05, section 13)
numeric_cols = ["yearpublished", "minplayers", "maxplayers", "minplaytime", "maxplaytime", "minage", "n_mechanics", "n_categories"]
gaps = games[numeric_cols].isna().sum(axis=1)
data = games[games["yearpublished"].notna() & (gaps < 4)].copy()           # drop no-year games and games missing 4+ of the 8 inputs
data["complete"] = data[numeric_cols].notna().all(axis=1)
tr_c, te_c = train_test_split(data[data["complete"]].reset_index(drop=True), test_size=0.3, random_state=1)    # = notebook 04's split
tr_i, te_i = train_test_split(data[~data["complete"]].reset_index(drop=True), test_size=0.3, random_state=1)
train_df, test_df = pd.concat([tr_c, tr_i], ignore_index=True), pd.concat([te_c, te_i], ignore_index=True)
y_train, y_test = train_df["average"], test_df["average"]

top_mech = train_df["boardgamemechanic"].str.get_dummies(sep="|").mean().sort_values(ascending=False).head(20).index
top_cat = train_df["boardgamecategory"].str.get_dummies(sep="|").mean().sort_values(ascending=False).head(15).index

def build_inputs(frame):
    X = frame[numeric_cols].copy()
    for col in ["minplayers", "maxplayers", "minplaytime", "maxplaytime"]:
        X[col] = np.log10(X[col])
    mech = frame["boardgamemechanic"].str.get_dummies(sep="|").reindex(columns=top_mech, fill_value=0).add_prefix("mech_")
    cat = frame["boardgamecategory"].str.get_dummies(sep="|").reindex(columns=top_cat, fill_value=0).add_prefix("cat_")
    flags = pd.DataFrame({"mech_list_missing": frame["boardgamemechanic"].isna().astype(int),
                          "cat_list_missing": frame["boardgamecategory"].isna().astype(int)}, index=frame.index)
    return pd.concat([X, mech, cat, flags], axis=1)

X_train = build_inputs(train_df)
X_test = build_inputs(test_df)
print("games used:", len(data), "| training rows:", len(train_df), "| test rows:", len(test_df), "| inputs:", X_train.shape[1])

# MODEL: task 10's deployed model (notebook 13): the regression tree tuned on task 1's rows, median imputer first
def make_model():
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("model", DecisionTreeRegressor(max_depth=12, min_samples_leaf=50, random_state=0))])
# ---- paste to here ----

assert DEPLOY == "tree", "the deploy decision changed: update the MODEL part of this cell"
assert X_train.equals(saved[0]) and X_test.equals(saved[1]) and test_df["id"].equals(saved[2])
assert np.allclose(make_model().fit(X_train, y_train).predict(X_test), deployed_pred)
print("the pasted cell rebuilds this notebook's rows, inputs and deployed model's predictions exactly")

## 11. What did not work, and notes

NOTES_PLACEHOLDER

### Post for the group chat

POST_PLACEHOLDER